# 00. Data Check — Cell2Cell 통신사 고객 이탈 예측

**목적**: 본격적인 EDA(`01_eda.ipynb`)에 들어가기 전에, 데이터가 어떻게 생겼는지
가볍게 훑어보고 "지저분한 부분"이 있는지 사실 위주로 확인한다. 해석·인사이트는
여기서 다루지 않고 `01_eda.ipynb`에서 다룬다.

**입력 데이터**: `data/raw/cell2celltrain.csv` (51,047명), `data/raw/cell2cellholdout.csv` (20,000명, Target 결측)


## 0. 환경 설정

In [3]:
import pandas as pd
from pathlib import Path

ROOT_DIR = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
TRAIN_PATH = ROOT_DIR / "data" / "raw" / "cell2celltrain.csv"
HOLDOUT_PATH = ROOT_DIR / "data" / "raw" / "cell2cellholdout.csv"

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)


## 1. 파일 로드

In [4]:
train_df = pd.read_csv(TRAIN_PATH)
holdout_df = pd.read_csv(HOLDOUT_PATH)

print("train shape:", train_df.shape)
print("holdout shape:", holdout_df.shape)


train shape: (51047, 58)
holdout shape: (20000, 58)


## 2. 컬럼 구성 확인 — train과 holdout이 같은 컬럼을 쓰는지

In [5]:
print("컬럼 수 - train:", len(train_df.columns), "/ holdout:", len(holdout_df.columns))
print("컬럼이 완전히 같은가:", list(train_df.columns) == list(holdout_df.columns))


컬럼 수 - train: 58 / holdout: 58
컬럼이 완전히 같은가: True


## 3. shape · dtype · 첫/끝 몇 행

In [6]:
train_df.dtypes


CustomerID                     int64
Churn                            str
MonthlyRevenue               float64
MonthlyMinutes               float64
TotalRecurringCharge         float64
DirectorAssistedCalls        float64
OverageMinutes               float64
RoamingCalls                 float64
PercChangeMinutes            float64
PercChangeRevenues           float64
DroppedCalls                 float64
BlockedCalls                 float64
UnansweredCalls              float64
CustomerCareCalls            float64
ThreewayCalls                float64
ReceivedCalls                float64
OutboundCalls                float64
InboundCalls                 float64
PeakCallsInOut               float64
OffPeakCallsInOut            float64
DroppedBlockedCalls          float64
CallForwardingCalls          float64
CallWaitingCalls             float64
MonthsInService                int64
UniqueSubs                     int64
ActiveSubs                     int64
ServiceArea                      str
H

In [7]:
train_df.head()


,CustomerID,Churn,MonthlyRevenue,MonthlyMinutes,TotalRecurringCharge,DirectorAssistedCalls,OverageMinutes,RoamingCalls,PercChangeMinutes,PercChangeRevenues,DroppedCalls,BlockedCalls,UnansweredCalls,CustomerCareCalls,ThreewayCalls,ReceivedCalls,OutboundCalls,InboundCalls,PeakCallsInOut,OffPeakCallsInOut,DroppedBlockedCalls,CallForwardingCalls,CallWaitingCalls,MonthsInService,UniqueSubs,ActiveSubs,ServiceArea,Handsets,HandsetModels,CurrentEquipmentDays,AgeHH1,AgeHH2,ChildrenInHH,HandsetRefurbished,HandsetWebCapable,TruckOwner,RVOwner,Homeownership,BuysViaMailOrder,RespondsToMailOffers,OptOutMailings,NonUSTravel,OwnsComputer,HasCreditCard,RetentionCalls,RetentionOffersAccepted,NewCellphoneUser,NotNewCellphoneUser,ReferralsMadeBySubscriber,IncomeGroup,OwnsMotorcycle,AdjustmentsToCreditRating,HandsetPrice,MadeCallToRetentionTeam,CreditRating,PrizmCode,Occupation,MaritalStatus
0,3000002,Yes,24.00,219.0,22.0,0.25,0.0,0.0,-157.0,-19.0,0.7,0.7,6.3,0.0,0.0,97.2,0.0,0.0,58.0,24.0,1.3,0.0,0.3,61,2,1,SEAPOR503,2.0,2.0,361.0,62.0,0.0,No,No,Yes,No,No,Known,Yes,Yes,No,No,Yes,Yes,1,0,No,No,0,4,No,0,30,Yes,1-Highest,Suburban,Professional,No
1,3000010,Yes,16.99,10.0,17.0,0.00,0.0,0.0,-4.0,0.0,0.3,0.0,2.7,0.0,0.0,0.0,0.0,0.0,5.0,1.0,0.3,0.0,0.0,58,1,1,PITHOM412,2.0,1.0,1504.0,40.0,42.0,Yes,No,No,No,No,Known,Yes,Yes,No,No,Yes,Yes,0,0,Yes,No,0,5,No,0,30,No,4-Medium,Suburban,Professional,Yes
2,3000014,No,38.00,8.0,38.0,0.00,0.0,0.0,-2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4,0.3,0.0,1.3,3.7,0.0,0.0,0.0,60,1,1,MILMIL414,1.0,1.0,1812.0,26.0,26.0,Yes,No,No,No,No,Unknown,No,No,No,No,No,Yes,0,0,Yes,No,0,6,No,0,Unknown,No,3-Good,Town,Crafts,Yes
3,3000022,No,82.28,1312.0,75.0,1.24,0.0,0.0,157.0,8.1,52.0,7.7,76.0,4.3,1.3,200.3,370.3,147.0,555.7,303.7,59.7,0.0,22.7,59,2,2,PITHOM412,9.0,4.0,458.0,30.0,0.0,No,No,Yes,No,No,Known,Yes,Yes,No,No,No,Yes,0,0,Yes,No,0,6,No,0,10,No,4-Medium,Other,Other,No
4,3000026,Yes,17.14,0.0,17.0,0.00,0.0,0.0,0.0,-0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,53,2,2,OKCTUL918,4.0,3.0,852.0,46.0,54.0,No,No,No,No,No,Known,Yes,Yes,No,No,Yes,Yes,0,0,No,Yes,0,9,No,1,10,No,1-Highest,Other,Professional,Yes


In [8]:
train_df.tail()


,CustomerID,Churn,MonthlyRevenue,MonthlyMinutes,TotalRecurringCharge,DirectorAssistedCalls,OverageMinutes,RoamingCalls,PercChangeMinutes,PercChangeRevenues,DroppedCalls,BlockedCalls,UnansweredCalls,CustomerCareCalls,ThreewayCalls,ReceivedCalls,OutboundCalls,InboundCalls,PeakCallsInOut,OffPeakCallsInOut,DroppedBlockedCalls,CallForwardingCalls,CallWaitingCalls,MonthsInService,UniqueSubs,ActiveSubs,ServiceArea,Handsets,HandsetModels,CurrentEquipmentDays,AgeHH1,AgeHH2,ChildrenInHH,HandsetRefurbished,HandsetWebCapable,TruckOwner,RVOwner,Homeownership,BuysViaMailOrder,RespondsToMailOffers,OptOutMailings,NonUSTravel,OwnsComputer,HasCreditCard,RetentionCalls,RetentionOffersAccepted,NewCellphoneUser,NotNewCellphoneUser,ReferralsMadeBySubscriber,IncomeGroup,OwnsMotorcycle,AdjustmentsToCreditRating,HandsetPrice,MadeCallToRetentionTeam,CreditRating,PrizmCode,Occupation,MaritalStatus
51042,3399958,Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.3,2.7,48.3,0.0,0.0,58.9,17.0,1.3,160.3,33.3,12.0,0.0,0.3,29,1,1,LAXSFN818,2.0,2.0,526.0,68.0,64.0,No,Yes,Yes,No,No,Known,Yes,Yes,No,No,No,Yes,0,0,No,No,0,6,No,0,60,No,1-Highest,Suburban,Other,Yes
51043,3399974,No,95.17,1745.0,85.0,0.99,45.0,4.7,122.0,15.9,16.7,0.7,41.3,0.0,0.0,681.5,89.7,33.3,318.7,248.3,17.3,0.0,14.3,29,1,1,LAXCDG310,2.0,2.0,464.0,48.0,48.0,Yes,No,Yes,No,No,Known,Yes,Yes,No,No,Yes,Yes,0,0,No,No,0,9,No,1,60,No,3-Good,Other,Other,No
51044,3399978,Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,71.7,4.3,287.0,1.3,13.7,1225.3,430.3,87.7,1359.0,910.3,76.0,0.0,6.7,25,1,1,LAXCDG310,3.0,2.0,378.0,36.0,0.0,No,No,Yes,No,No,Known,No,No,No,No,No,Yes,0,0,No,No,0,7,No,1,80,No,5-Low,Other,Clerical,No
51045,3399990,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,31,1,1,NEVPOW619,2.0,2.0,433.0,32.0,0.0,Yes,No,Yes,No,No,Unknown,No,No,No,No,No,No,0,0,No,No,0,9,No,0,30,No,5-Low,Other,Other,No
51046,3399994,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.7,0.7,117.3,1.3,1.0,330.9,55.0,16.7,442.3,167.3,19.3,0.0,0.0,25,1,1,NEVPOW619,7.0,5.0,75.0,0.0,0.0,No,No,Yes,No,No,Unknown,No,No,No,No,No,No,2,1,No,No,0,0,No,1,60,Yes,5-Low,Other,Other,Unknown


## 4. Target(Churn) 컬럼 확인

In [9]:
print("train의 Churn 분포:")
print(train_df['Churn'].value_counts(dropna=False))
print()
print("holdout의 Churn 분포 (Target 없는 파일인지 확인):")
print(holdout_df['Churn'].value_counts(dropna=False))


train의 Churn 분포:
Churn
No     36336
Yes    14711
Name: count, dtype: int64

holdout의 Churn 분포 (Target 없는 파일인지 확인):
Churn
NaN    20000
Name: count, dtype: int64


**확인 결과**: `holdout_df['Churn']`이 전부 결측(NaN)이면 이 파일은 Test로 쓸 수
없다는 뜻 — 팀 작업계획서 방침대로 Streamlit 시연 전용으로만 사용한다.

## 5. 중복 확인

In [10]:
print("CustomerID 중복 수(train):", train_df['CustomerID'].duplicated().sum())
print("전체 행 중복 수(train):", train_df.duplicated().sum())
print("CustomerID 중복 수(holdout):", holdout_df['CustomerID'].duplicated().sum())
print("train·holdout 간 CustomerID 겹침:",
      set(train_df['CustomerID']) & set(holdout_df['CustomerID']))


CustomerID 중복 수(train): 0
전체 행 중복 수(train): 0
CustomerID 중복 수(holdout): 0
train·holdout 간 CustomerID 겹침: set()


## 6. 결측치 현황

In [11]:
null_counts = train_df.isnull().sum()
null_ratio = (null_counts / len(train_df) * 100).round(2)
missing_summary = pd.DataFrame({'결측수': null_counts, '결측비율(%)': null_ratio})
missing_summary[missing_summary['결측수'] > 0].sort_values('결측수', ascending=False)


,결측수,결측비율(%)
AgeHH2,909,1.78
AgeHH1,909,1.78
PercChangeRevenues,367,0.72
PercChangeMinutes,367,0.72
MonthlyRevenue,156,0.31
MonthlyMinutes,156,0.31
RoamingCalls,156,0.31
OverageMinutes,156,0.31
DirectorAssistedCalls,156,0.31
TotalRecurringCharge,156,0.31


## 7. 수치형 컬럼 기초 통계 (describe) — 불가능한 값 확인용

In [12]:
num_cols = train_df.select_dtypes(include='number').columns.drop('CustomerID')
train_df[num_cols].describe().T


,count,mean,std,min,25%,50%,75%,max
MonthlyRevenue,50891.0,58.834492,44.507336,-6.17,33.61,48.46,71.065,1223.38
MonthlyMinutes,50891.0,525.653416,529.871063,0.00,158.00,366.00,723.000,7359.00
TotalRecurringCharge,50891.0,46.830088,23.848871,-11.00,30.00,45.00,60.000,400.00
DirectorAssistedCalls,50891.0,0.895229,2.228546,0.00,0.00,0.25,0.990,159.39
OverageMinutes,50891.0,40.027785,96.588076,0.00,0.00,3.00,41.000,4321.00
RoamingCalls,50891.0,1.236244,9.818294,0.00,0.00,0.00,0.300,1112.40
PercChangeMinutes,50680.0,-11.547908,257.514772,-3875.00,-83.00,-5.00,66.000,5192.00
PercChangeRevenues,50680.0,-1.191985,39.574915,-1107.70,-7.10,-0.30,1.600,2483.50
DroppedCalls,51047.0,6.011489,9.043955,0.00,0.70,3.00,7.700,221.70
BlockedCalls,51047.0,4.085672,10.946905,0.00,0.00,1.00,3.700,384.30


min 값이 음수인 컬럼이 있는지, max 값이 비현실적으로 큰 컬럼이 있는지를 눈으로
훑어본다. (자세한 이상치 판단·처리는 `02_eda.ipynb`와 `src/data.py`에서 다룬다)

## 8. 눈에 띄는 '지저분한' 컬럼 3개 — 미리 확인만

전처리에서 별도 처리가 필요했던 컬럼들이 실제로 어떤 모습인지 여기서 먼저 확인한다.


In [13]:
# 8-1. ServiceArea: 고유값이 너무 많지 않은지
print("ServiceArea 고유값 수:", train_df['ServiceArea'].nunique())
train_df['ServiceArea'].value_counts().head(10)


ServiceArea 고유값 수: 747


ServiceArea
NYCBRO917    1684
HOUHOU281    1510
DALDAL214    1498
NYCMAN917    1182
APCFCH703     783
DALFTW817     782
SANSAN210     724
APCSIL301     670
SANAUS512     612
SFROAK510     605
Name: count, dtype: int64

In [14]:
# 8-2. HandsetPrice: 숫자와 문자가 섞여 있는지
print("HandsetPrice dtype:", train_df['HandsetPrice'].dtype)
train_df['HandsetPrice'].value_counts().head(10)


HandsetPrice dtype: str


HandsetPrice
Unknown    28982
30          7328
150         4115
130         2105
80          1960
10          1928
60          1776
200         1266
100         1235
40           249
Name: count, dtype: int64

In [15]:
# 8-3. AgeHH1 / AgeHH2: 0이 '결측'인지 '정상값'인지 확인
print("AgeHH1 == 0 개수:", (train_df['AgeHH1'] == 0).sum())
print("AgeHH1 결측(NaN) 개수:", train_df['AgeHH1'].isnull().sum())
print("AgeHH1/AgeHH2 항상 같이 결측인가:",
      (train_df['AgeHH1'].isnull() == train_df['AgeHH2'].isnull()).all())


AgeHH1 == 0 개수: 13917
AgeHH1 결측(NaN) 개수: 909
AgeHH1/AgeHH2 항상 같이 결측인가: True


## 9. 요약 — 다음 단계(01_eda.ipynb)로 넘길 체크리스트

- [x] train/holdout 컬럼 구성 동일함 확인
- [x] holdout은 Target 전부 결측 → Test 불가, 시연 전용으로만 사용
- [x] CustomerID 중복 없음, train·holdout 간 고객 겹침 없음
- [x] 결측치 14개 컬럼, 최대 비율 1.78% → 처리 방법은 `01_eda.ipynb`·`src/data.py`에서 결정
- [x] ServiceArea 고유값 747개 → 축약 필요
- [x] HandsetPrice에 'Unknown' 문자열 혼재 → 숫자 변환 필요
- [x] AgeHH1/AgeHH2는 0이 이미 '세대원 없음'을 의미하는 센티널로 쓰이고 있음 → 결측 대체 시 유의

이 사실들을 바탕으로 `01_eda.ipynb`에서 본격적인 이탈률 분석·시각화·인사이트 도출을 진행한다.
